# ChromaDB local use case

This notebook connects to a local Chroma server (Docker or `chroma run`) and demonstrates adding and querying data.

If you are not running a server, see the embedded alternative near the end.

## Connection settings

This notebook targets the Docker container port mapping (`localhost:8001` by default).
If your server is on a different port, set an env var before running this notebook:

```
export CHROMA_PORT=8001
```

In [3]:
import os
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
print("Heartbeat:", client.heartbeat())

Heartbeat: 1770716820588917334


In [4]:
collection = client.get_or_create_collection("local_demo")

collection.add(
    ids=["doc-1", "doc-2", "doc-3"],
    documents=[
        "Chroma is a vector database for embeddings.",
        "A vector search lets you find similar meaning.",
        "macOS users can run Chroma locally using Docker or chroma run.",
    ],
    metadatas=[
        {"source": "docs", "topic": "chroma"},
        {"source": "docs", "topic": "search"},
        {"source": "docs", "topic": "mac"},
    ],
)

print("Count:", collection.count())

Count: 3


In [5]:
results = collection.query(
    query_texts=["How do I run Chroma locally on a Mac?"],
    n_results=2,
)
results

{'ids': [['doc-3', 'doc-1']],
 'distances': [[0.3023473, 1.2297055]],
 'embeddings': None,
 'metadatas': [[{'topic': 'mac', 'source': 'docs'},
   {'source': 'docs', 'topic': 'chroma'}]],
 'documents': [['macOS users can run Chroma locally using Docker or chroma run.',
   'Chroma is a vector database for embeddings.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [6]:
# Read back specific documents by id
collection.get(ids=["doc-1", "doc-3"])

{'ids': ['doc-1', 'doc-3'],
 'embeddings': None,
 'metadatas': [{'source': 'docs', 'topic': 'chroma'},
  {'source': 'docs', 'topic': 'mac'}],
 'documents': ['Chroma is a vector database for embeddings.',
  'macOS users can run Chroma locally using Docker or chroma run.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

## Optional: Embedded mode (no server)

If you are not running a server, use the in-process client instead:

In [7]:
from chromadb import PersistentClient

embedded_client = PersistentClient(path="./chroma_data")
embedded_collection = embedded_client.get_or_create_collection("local_demo")
print("Embedded count:", embedded_collection.count())

Embedded count: 0


In [8]:
## test 
import os
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))
client = chromadb.HttpClient(host=HOST, port=PORT)
ids = collection.get(ids=["doc-1", "doc-3"])
print(ids)

{'ids': ['doc-1', 'doc-3'], 'embeddings': None, 'metadatas': [{'source': 'docs', 'topic': 'chroma'}, {'source': 'docs', 'topic': 'mac'}], 'documents': ['Chroma is a vector database for embeddings.', 'macOS users can run Chroma locally using Docker or chroma run.'], 'data': None, 'uris': None, 'included': ['metadatas', 'documents']}


In [ ]:
## My Examples from here 

In [19]:
## 1) Collections - 
### Concept: A collection is a named container for vectors + documents + metadata.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.count()

5

In [11]:
## 2) Documents and IDs
### Concept: Each document must have a unique ID.


3

In [13]:
## 3) Metadata
### Concept: Metadata lets you filter search results.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.add(
    ids=["d4", "d5"],
    documents=[
        "Mac users can run Chroma with Docker.",
        "Server mode uses chroma run.",
    ],
    metadatas=[
        {"topic": "mac", "source": "docs"},
        {"topic": "server", "source": "docs"},
    ],
)

In [14]:
## 4) Similarity Search (Query)
### import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.query(
    query_texts=["How do I run Chroma locally?"],
    n_results=2,
)

{'ids': [['d5', 'd4']],
 'distances': [[0.63203347, 0.64515984]],
 'embeddings': None,
 'metadatas': [[{'source': 'docs', 'topic': 'server'},
   {'topic': 'mac', 'source': 'docs'}]],
 'documents': [['Server mode uses chroma run.',
   'Mac users can run Chroma with Docker.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [16]:
## 5) Metadata Filtering
### Concept: Filter search by metadata fields.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.query(
    query_texts=["How do I run Chroma locally?"],
    n_results=3,
    where={"topic": "mac"},
)

{'ids': [['d4']],
 'distances': [[0.64515984]],
 'embeddings': None,
 'metadatas': [[{'topic': 'mac', 'source': 'docs'}]],
 'documents': [['Mac users can run Chroma with Docker.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [17]:
## 6) Get by ID
### Concept: Fetch stored documents directly by ID
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.get(ids=["d1", "d4"])

{'ids': ['d1', 'd4'],
 'embeddings': None,
 'metadatas': [None, {'source': 'docs', 'topic': 'mac'}],
 'documents': ['Chroma is a vector database.',
  'Mac users can run Chroma with Docker.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

In [18]:
## 7) Update Documents
### Concept: Update a document by re-adding with the same ID.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.add(
    ids=["d2"],
    documents=["Vectors represent meaning and context."],
    metadatas=[{"topic": "vectors"}],
)
collection.get(ids=["d2"])

{'ids': ['d2'],
 'embeddings': None,
 'metadatas': [None],
 'documents': ['Vectors represent meaning.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

In [21]:
## 8) Delete Documents
### Concept: Remove documents by ID.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.delete(ids=["d3"])
collection.count()    

4

In [22]:
## 9) Collections List and Delete
### Concept: You can list and remove entire collections.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
client.list_collections()

[Collection(name=concepts_demo), Collection(name=local_demo)]

In [23]:
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
client.delete_collection("concepts_demo")
client.list_collections()

[Collection(name=local_demo)]

In [24]:
## 11) Embeddings (What happens behind the scenes)
### Concept: Chroma stores vectors computed from text. If you pass raw text, Chroma uses its default embedding function unless configured otherwise.

### You can also supply your own embeddings (example uses dummy vectors):
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.add(
    ids=["e1", "e2"],
    documents=["alpha", "beta"],
    embeddings=[[0.1, 0.2, 0.3], [0.2, 0.1, 0.0]],
)